In [ ]:
"""
Path Agreement Evaluation
Compares model-generated supporting evidence paths against
human-annotated gold evidence paths from the gold dataset.

Merging strategy: doc_id + normalised triplet_key (from summary_triplet_full)
Uses your exact normalize_triplet function.

Metrics:
  - Exact match rate
  - Jaccard similarity
  - Precision (model paths that match human paths)
  - Recall (human paths the model found)
  - F1 score

Stratified by confidence tier:
  - All supported (conf 1 + 2)
  - Direct support (conf 2 only)
  - Inferential support (conf 1 only)
  - Unsupported (conf 0, diagnostic)

UPDATE the file paths below before running.
"""

import ast
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# # ── Configuration — UPDATE THESE ──────────────────────────────────────────────
# GOLD_FILE = "/path/to/gold_dataset.csv"

# MODEL_FILES = {
#     "Claude Sonnet 4":  "/path/to/sonnet_outputs.csv",
#     "Qwen3 8B LoRA":    "/path/to/lora_outputs.csv",
#     "Qwen3 8B base":    "/path/to/base_outputs.csv",
#     "Qwen2.5 7B LoRA":  "/path/to/qwen25_outputs.csv",
# }

# Column in model output files containing supporting evidence.
# If your model output stores the full JSON dict in one column
# (e.g. "fact_eval_res_lora"), the script will auto-extract
# the "supporting_evidence" field from it.
MODEL_EVIDENCE_COL = "supporting_evidence"

# OUT = "/mnt/user-data/outputs/"
COLOURS = ["#3b82f6", "#10b981", "#f59e0b", "#ef4444"]

plt.rcParams.update({
    "font.family": "serif", "font.size": 10,
    "axes.titlesize": 11, "savefig.dpi": 300, "savefig.bbox": "tight",
})


# ── Your exact normalize_triplet function ──────────────────────────────────────

def normalize_triplet(x):
    if pd.isna(x):
        return tuple()
    if isinstance(x, str):
        try:
            x = ast.literal_eval(x)
        except (ValueError, SyntaxError):
            return tuple([x.strip()])

    normalized_items = []
    if isinstance(x, (list, tuple)):
        if all(isinstance(elem, (list, tuple)) for elem in x) and len(x) == 1:
            x = x[0]
            if not isinstance(x, (list, tuple)):
                x = [x]
        for item in x:
            normalized_items.append(str(item).strip())
    else:
        normalized_items.append(str(x).strip())

    return tuple(normalized_items)


# ── Evidence parsing ───────────────────────────────────────────────────────────

def parse_evidence_list(raw):
    """Parse supporting evidence into a list of 3-element tuples."""
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return []
    if isinstance(raw, list):
        return [tuple(str(x).strip() for x in t)
                for t in raw if isinstance(t, (list, tuple)) and len(t) == 3]
    if isinstance(raw, str):
        raw = raw.strip()
        if not raw or raw in ("[]", "nan", "None", ""):
            return []
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(raw)
                if isinstance(parsed, dict):
                    parsed = parsed.get("supporting_evidence", [])
                return [tuple(str(x).strip() for x in t)
                        for t in parsed
                        if isinstance(t, (list, tuple)) and len(t) == 3]
            except Exception:
                continue
    return []


def extract_model_evidence(raw):
    """
    Extract supporting_evidence from model output column.
    Handles: direct list, JSON string, JSON dict with nested field.
    """
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return []
    if isinstance(raw, list):
        return parse_evidence_list(raw)
    if isinstance(raw, str):
        raw = raw.strip()
        if not raw:
            return []
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(raw)
                if isinstance(parsed, dict):
                    return parse_evidence_list(
                        parsed.get("supporting_evidence", [])
                    )
                return parse_evidence_list(parsed)
            except Exception:
                continue
    return []


def normalise_set(evidence_list):
    """Lowercase + strip evidence triplets for fuzzy set comparison."""
    return set(
        tuple(x.lower().strip() for x in t)
        for t in evidence_list if len(t) == 3
    )


# ── Metrics ───────────────────────────────────────────────────────────────────

def compute_metrics(gold_set, model_set):
    if not gold_set and not model_set:
        return None
    inter = gold_set & model_set
    union = gold_set | model_set
    prec  = len(inter) / len(model_set) if model_set else 0.0
    rec   = len(inter) / len(gold_set)  if gold_set  else 0.0
    return {
        "exact_match": float(gold_set == model_set),
        "jaccard":     len(inter) / len(union) if union else 0.0,
        "precision":   prec,
        "recall":      rec,
        "f1":          2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0,
        "gold_size":   len(gold_set),
        "model_size":  len(model_set),
        "n_overlap":   len(inter),
    }


# ── Load gold ──────────────────────────────────────────────────────────────────

def load_gold(path):
    gold = pd.read_csv(path)

    # Merging key
    gold["triplet_key"] = gold["summary_triplet_full"].apply(normalize_triplet)

    # Gold evidence
    gold["gold_evidence"] = gold["supporting_evidene_combined"].apply(
        parse_evidence_list
    )

    # Confidence (handles "2 - direct evidence" style)
    conf_col = ("Your Annotation.1"
                if "Your Annotation.1" in gold.columns else "confidence")
    def parse_conf(x):
        if pd.isna(x): return None
        for ch in str(x):
            if ch.isdigit(): return int(ch)
        return None
    gold["conf_int"] = gold[conf_col].apply(parse_conf)

    # is_supported boolean
    sup_col = ("Your_annotation_is_supported"
               if "Your_annotation_is_supported" in gold.columns
               else "is_supported")
    gold["is_supported_bool"] = gold[sup_col].apply(
        lambda x: str(x).strip().upper() == "TRUE" if pd.notna(x) else None
    )

    print(f"Gold loaded: {len(gold)} rows")
    print(f"  Confidence distribution: "
          f"{gold['conf_int'].value_counts().sort_index().to_dict()}")
    return gold


# ── Load model and merge ───────────────────────────────────────────────────────

def load_and_merge(gold_df, model_path, model_name):
    # Load model output
    if model_path.endswith((".json", ".jsonl")):
        rows = []
        with open(model_path) as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        data_df = pd.DataFrame(rows)
    else:
        data_df = pd.read_csv(model_path)

    # Triplet key — same function as gold
    triplet_col = next(
        (c for c in ["summary_triplet_full", "summary_triple"]
         if c in data_df.columns), None
    )
    if triplet_col is None:
        raise ValueError(f"No triplet column in {model_name}. "
                         f"Columns: {list(data_df.columns)}")
    data_df["triplet_key"] = data_df[triplet_col].apply(normalize_triplet)

    # Model evidence
    ev_col = MODEL_EVIDENCE_COL
    if ev_col not in data_df.columns:
        # Try common alternatives
        for alt in ["fact_eval_res_lora", "fact_eval_res", "output",
                    "fact_eval_res_base", "fact_eval_res_sonnet"]:
            if alt in data_df.columns:
                ev_col = alt
                print(f"  Using column '{alt}' for model evidence")
                break
        else:
            raise ValueError(
                f"Cannot find evidence column in {model_name}. "
                f"Columns: {list(data_df.columns)}"
            )
    data_df["model_evidence"] = data_df[ev_col].apply(extract_model_evidence)

    # ── YOUR MERGING STRATEGY ─────────────────────────────────────────────────
    merged = pd.merge(
        gold_df,
        data_df[["doc_id", "triplet_key", "model_evidence"]],
        on=["doc_id", "triplet_key"],
        suffixes=("_gold", "_data"),
        how="inner"
    )

    print(f"{model_name}: {len(merged)} matched rows "
          f"(gold={len(gold_df)}, model={len(data_df)}, "
          f"unmatched gold rows={len(gold_df) - len(merged)})")

    return merged


# ── Evaluate ───────────────────────────────────────────────────────────────────

def evaluate(merged_df, model_name):
    rows = []
    for _, row in merged_df.iterrows():
        gold_set  = normalise_set(row["gold_evidence"])
        model_set = normalise_set(row["model_evidence"])
        m = compute_metrics(gold_set, model_set)
        if m is None:
            continue
        m["model"]     = model_name
        m["conf"]      = row["conf_int"]
        m["supported"] = row["is_supported_bool"]
        rows.append(m)
    return pd.DataFrame(rows)


# ── Aggregate ──────────────────────────────────────────────────────────────────

METRIC_COLS = ["exact_match", "jaccard", "precision", "recall", "f1"]

SUBSETS = [
    ("All supported",        lambda df: df[df["supported"] == True]),
    ("Direct (conf 2)",      lambda df: df[df["conf"] == 2]),
    ("Inferential (conf 1)", lambda df: df[df["conf"] == 1]),
    ("Unsupported (conf 0)", lambda df: df[df["conf"] == 0]),
]


def aggregate(results_df, model_name):
    print(f"\n{'='*60}\nMODEL: {model_name}\n{'='*60}")
    agg_rows = []
    for label, fn in SUBSETS:
        subset = fn(results_df)
        if len(subset) == 0:
            continue
        row = {"subset": label, "n": len(subset)}
        for m in METRIC_COLS:
            row[m]         = subset[m].mean()
            row[f"{m}_sd"] = subset[m].std()
        agg_rows.append(row)
        print(f"\n  {label} (n={len(subset)}):")
        for m in METRIC_COLS:
            print(f"    {m:15s}: {subset[m].mean():.4f} "
                  f"(SD={subset[m].std():.4f})")
    return pd.DataFrame(agg_rows)


# ── Figures ────────────────────────────────────────────────────────────────────

def plot_metric(all_agg, metric):
    models  = list(all_agg.keys())
    subsets = ["All supported", "Direct (conf 2)", "Inferential (conf 1)"]
    x       = np.arange(len(subsets))
    width   = 0.8 / len(models)

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, (model, agg_df) in enumerate(all_agg.items()):
        vals = []
        errs = []
        for s in subsets:
            row = agg_df[agg_df["subset"] == s]
            vals.append(row[metric].values[0]         if len(row) > 0 else 0)
            errs.append(row[f"{metric}_sd"].values[0] if len(row) > 0 else 0)

        offset = (i - len(models) / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width * 0.88,
                      label=model, color=COLOURS[i % len(COLOURS)],
                      alpha=0.85, edgecolor="white", zorder=2)
        ax.errorbar(x + offset, vals, yerr=errs,
                    fmt="none", color="#374151",
                    capsize=3, linewidth=1.2, zorder=3)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.008,
                    f"{val:.3f}", ha="center", va="bottom", fontsize=7.5)

    ax.set_xticks(x)
    ax.set_xticklabels(subsets, fontsize=9)
    ax.set_ylim(0, min(1.05, max(
        agg_df[metric].max() for agg_df in all_agg.values()
    ) * 1.25))
    ax.set_ylabel(f"Mean {metric.upper()}", labelpad=8)
    ax.set_title(
        f"Path Agreement — {metric.upper()} by Model and Confidence Tier\n"
        f"(Model evidence vs Human-annotated gold paths, error bars = ±1 SD)",
        fontweight="bold", pad=10
    )
    ax.legend(fontsize=8.5, loc="upper right")
    ax.grid(axis="y", alpha=0.3, zorder=1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(OUT + f"path_agreement_{metric}.pdf")
    plt.savefig(OUT + f"path_agreement_{metric}.png", dpi=200)
    plt.close()
    print(f"Saved: path_agreement_{metric}.pdf")


# ── LaTeX table ────────────────────────────────────────────────────────────────

def print_latex_table(all_agg):
    models = list(all_agg.keys())
    subsets_order = ["All supported", "Direct (conf 2)", "Inferential (conf 1)"]

    print("\n" + "="*70)
    print("LaTeX TABLE")
    print("="*70)
    print(r"\begin{table}[h]")
    print(r"\centering\small")
    print(r"\begin{tabular}{llccccc}")
    print(r"\hline")
    print(r"\textbf{Model} & \textbf{Subset} & \textbf{n} & "
          r"\textbf{Jaccard} & \textbf{Prec.} & "
          r"\textbf{Rec.} & \textbf{F1} \\")
    print(r"\hline")

    for model in models:
        agg_df = all_agg[model]
        first  = True
        for s in subsets_order:
            row = agg_df[agg_df["subset"] == s]
            if len(row) == 0:
                continue
            model_label = r"\textbf{" + model + "}" if first else ""
            n   = int(row["n"].values[0])
            vals = " & ".join(
                f"{row[m].values[0]:.3f}"
                for m in ["jaccard", "precision", "recall", "f1"]
            )
            print(f"{model_label} & {s} & {n} & {vals} \\\\")
            first = False
        print(r"\hline")

    print(r"\end{tabular}")
    print(r"\caption{Path agreement between model-generated and "
          r"human-annotated evidence paths on the gold dataset. "
          r"Precision = proportion of model-cited paths matching human "
          r"annotation; Recall = proportion of human-annotated paths "
          r"found by the model; F1 = harmonic mean. Evaluated separately "
          r"for directly (confidence 2) and inferentially (confidence 1) "
          r"supported cases.}")
    print(r"\label{tab:path_agreement}")
    print(r"\end{table}")


# ── Main ───────────────────────────────────────────────────────────────────────

def main():
    print("Loading gold dataset...")
    gold_df = load_gold(GOLD_FILE)

    all_results, all_agg = {}, {}

    for model_name, model_path in MODEL_FILES.items():
        print(f"\n{'─'*50}")
        print(f"Processing: {model_name}")
        try:
            merged  = load_and_merge(gold_df, model_path, model_name)
            results = evaluate(merged, model_name)
            agg     = aggregate(results, model_name)
            all_results[model_name] = results
            all_agg[model_name]     = agg
        except FileNotFoundError:
            print(f"  SKIPPED — file not found: {model_path}")
        except Exception as e:
            print(f"  ERROR: {e}")
            import traceback; traceback.print_exc()

    if all_agg:
        print("\nGenerating figures...")
        for metric in ["f1", "jaccard", "recall", "precision"]:
            plot_metric(all_agg, metric)
        print_latex_table(all_agg)

    print("\n✓ Done.")
    return all_results, all_agg


# if __name__ == "__main__":
#     all_results, all_agg = main()

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
GOLD_FILE = "/content/drive/MyDrive/research/gold_data/expanded_triplet_evaluations_annotated_evidence.csv"  # UPDATE THIS

# Map model name → path to model output CSV/JSON
# Each file must have: summary_triplet_full, supporting_evidence, confidence (or is_supported)
MODEL_FILES = {
    "Claude Sonnet 4":  "/content/drive/MyDrive/research/data/inferencing/claude_4_sonnet/expanded_triplet_evaluations_4.csv",   # UPDATE
    "Qwen3 8B LoRA":    "/content/drive/MyDrive/research/data/inferencing/lora_inference/qwen3/27-02-26/results_traversal_ck_300_t_01_v2.csv",     # UPDATE
    "Qwen3 8B base":    "/content/drive/MyDrive/research/data/inferencing/qwen3/03-04-26/results_traversal_qwen3_8b_t_01_expanded.csv",     # UPDATE
    "Qwen2.5 7B LoRA":  "/content/drive/MyDrive/research/data/inferencing/lora_inference/qwen2_5/08-04-26/results_traversal_ck_300_t_01_expanded.csv",   # UPDATE
    "Qwen3 8B LoRA No Algo" : "/content/drive/MyDrive/research/data/inferencing/lora_inference/qwen3/27-02-26/results_traversal_ck_300_t_01_no_algo_expanded.csv"
}

OUT = "/content/drive/MyDrive/research/evidence_path_asses/"

if __name__ == "__main__":
    all_results, all_agg = main()


Loading gold dataset...
Gold loaded: 623 rows
  Confidence distribution: {0.0: 122, 1.0: 111, 2.0: 238}

──────────────────────────────────────────────────
Processing: Claude Sonnet 4
Claude Sonnet 4: 622 matched rows (gold=623, model=622, unmatched gold rows=1)

MODEL: Claude Sonnet 4

  All supported (n=437):
    exact_match    : 0.3066 (SD=0.4616)
    jaccard        : 0.6304 (SD=0.2954)
    precision      : 0.7872 (SD=0.2754)
    recall         : 0.7148 (SD=0.2753)
    f1             : 0.7284 (SD=0.2547)

  Direct (conf 2) (n=238):
    exact_match    : 0.2815 (SD=0.4507)
    jaccard        : 0.6147 (SD=0.2969)
    precision      : 0.7735 (SD=0.2856)
    recall         : 0.7129 (SD=0.2877)
    f1             : 0.7149 (SD=0.2603)

  Inferential (conf 1) (n=111):
    exact_match    : 0.2613 (SD=0.4413)
    jaccard        : 0.6118 (SD=0.2844)
    precision      : 0.7569 (SD=0.2707)
    recall         : 0.7002 (SD=0.2566)
    f1             : 0.7169 (SD=0.2489)

  Unsupported (conf 0) (n